# Production VAD Pipeline (UL-UNAS Denoising & Silero VAD)

This notebook visualizes and evaluates the **production Voice Activity Detection (VAD)** pipeline from `backend.pipeline.segmentation.audio.vad`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/segmentation/silero_and_ul_unas_wet_dry_VAD_mixture.ipynb)

### Key Features:
* **Production Implementation**: Directly imports and executes `VoiceActivityDetector` from the repository codebase (`backend.pipeline.segmentation`), ensuring 100% consistency with production streaming workers.
* **UL-UNAS Sequence Denoiser**: Applies the streaming neural denoiser to eliminate background static and noise floor.
* **Wet/Dry Audio Mixture**: Blends the deep-learning denoiser ("wet" signal) with the bandpass-filtered audio ("dry" signal) controlled by an adjustable `blend_ratio`.
* **Spectral Gating & Tone Rejection**: Incorporates formant energy distribution checks (`_is_speech_segment`) and dual-peak neighborhood concentration checks (`_is_tone_segment`) to reject subaudible ticks and signaling tones (Quik-Call II / EAS).
* **Interactive Parameters**: All pipeline hyperparameters (bandpass cutoffs, denoiser blend ratio, presence boost EQ, VAD onset/offset thresholds, padding) can be dynamically adjusted using Colab `@param` form controls.
* **Visualization & Evaluation**: Includes interactive WaveSurfer.js audio players, Silero VAD probability confidence plots, and automated frame-based evaluation against ground-truth benchmarks.

In [ ]:
%pip install -q onnxruntime pedalboard soundfile numba matplotlib

In [ ]:
# @title Repository Setup & Production Imports
import base64
import io
import math
import os
import pathlib
import subprocess
import tempfile
import wave
from pathlib import Path

from IPython.display import Audio, HTML, display
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

try:
    from google.colab import files

    HAS_COLAB_FILES = True
except ImportError:
    HAS_COLAB_FILES = False

# Resolve repository root and add to sys.path
current_path = Path.cwd().resolve()
repo_root = current_path
while repo_root.name and repo_root.name != "radio-transcription":
    if repo_root == repo_root.parent:
        break
    repo_root = repo_root.parent

if (repo_root / "backend").exists():
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    print(f"Repository root verified at: {repo_root}")
else:
    # If in standalone Google Colab without git clone
    if not (Path.cwd() / "backend").exists():
        print("Cloning repository into Colab environment...")
        !git clone -q https://github.com/watch-duty/radio-transcription.git
        if str(Path.cwd() / "radio-transcription") not in sys.path:
            sys.path.insert(0, str(Path.cwd() / "radio-transcription"))

from backend.pipeline.segmentation.audio.vad import VoiceActivityDetector
from backend.pipeline.segmentation.audio.dsp import TorchaudioHannResampler
from backend.pipeline.segmentation.constants import TARGET_SAMPLE_RATE

In [ ]:
# @title Verify & Download Production ONNX Models
from backend.pipeline.segmentation.audio.vad import MODELS_DIR

MODELS_DIR.mkdir(parents=True, exist_ok=True)
silero_path = MODELS_DIR / "silero_vad.onnx"
ulunas_path = MODELS_DIR / "ulunas_stft_sequence.onnx"

if not silero_path.exists():
    print(f"Downloading Silero VAD model to {silero_path}...")
    !wget -q -O "{silero_path}" https://raw.githubusercontent.com/snakers4/silero-vad/v6.2.1/src/silero_vad/data/silero_vad.onnx

if not ulunas_path.exists():
    print(f"Downloading UL-UNAS sequence denoiser model to {ulunas_path}...")
    !wget -q -O "{ulunas_path}" https://raw.githubusercontent.com/watch-duty/radio-transcription/main/backend/pipeline/segmentation/audio/models/ulunas_stft_sequence.onnx

print(f"ONNX models verified in: {MODELS_DIR}")

In [ ]:
# @title Constants

GROUND_TRUTH_SEGMENTS = {
    "test_stress.flac": [(0.4, 2.85)],
    "test_joined.flac": [(8.3, 10.7), (12.3, 15.6), (20.3, 23.0), (26.2, 27.0)],
    "test_bcfy.flac": [(0.0, 1.1), (1.95, 5.3), (7.25, 10.9), (11.6, 12.2)],
    "test_dispatch_amador.flac": [
        (2.7, 12.5),
        (14.4, 15.8),
        (17.5, 24.6),
        (27.3, 29.7),
        (31.4, 33.7),
        (38.1, 40.5),
        (47.2, 49.4),
        (56.2, 60.6),
        (62.6, 65.3),
    ],
    "test_dispatch_sku.flac": [
        (0.420, 2.593),
        (3.3, 5.788),
        (6.242, 8.838),
        (8.861, 11.044),
        (11.691, 14.717),
        (14.811, 17.014),
        (17.781, 19.707),
        (20.253, 22.040),
        (22.843, 24.669),
        (25.547, 27.728),
        (28.471, 29.830),
        (30.845, 32.907),
        (33.003, 34.615),
        (35.704, 37.877),
        (40.570, 41.772),
        (42.467, 44.470),
        (45.874, 49.212),
        (49.373, 51.884),
        (52.768, 54.178),
    ],
    "test_middlebury_quiet_segments.mp3": [(0.47, 1.4), (3.9, 6.4)],
    "test_quiet_speech_loud_transient.mp3": [(0.213, 0.8), (2.037, 3.869)],
    "test_middlebury_quiet_spiky.mp3": [(0.18, 1.45)],
    "test_only_static_middlebury.mp3": [],
    "test_tone_only.flac": [],
    "test_subaudible_flickering.flac": [],  # Open-squelch static ticks and 72Hz flickering interference (100% rejected as non-speech)
}

In [ ]:
# @title Visualization & Evaluation Helpers

INT16_MAX_FLOAT = 32768.0


def load_and_resample_audio(
    audio_filepath: str, target_sr: int = TARGET_SAMPLE_RATE
) -> np.ndarray:
    """Loads an audio file via soundfile/ffmpeg and resamples to target_sr (16 kHz)."""
    is_mp3 = str(audio_filepath).lower().endswith(".mp3")
    use_fallback = is_mp3

    if not is_mp3:
        try:
            audio_data, orig_sr = sf.read(audio_filepath, always_2d=True)
            audio_data = audio_data.mean(axis=1).astype(np.float32)
        except Exception as e:
            print(
                f"[load_and_resample_audio] soundfile failed ({e}), falling back to ffmpeg..."
            )
            use_fallback = True

    if use_fallback:
        probe_cmd = [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "stream=sample_rate",
            "-of",
            "default=noprint_wrappers=1:nokey=1",
            str(audio_filepath),
        ]
        orig_sr = int(
            subprocess.check_output(probe_cmd)
            .decode("utf-8")
            .strip()
            .split("\n")[0]
        )

        command = [
            "ffmpeg",
            "-i",
            str(audio_filepath),
            "-f",
            "f32le",
            "-acodec",
            "pcm_f32le",
            "-ac",
            "1",
            "-",
        ]
        pipe = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL
        )
        out, _ = pipe.communicate()
        audio_data = np.frombuffer(out, dtype=np.float32)

    if orig_sr != target_sr:
        print(
            f"[load_and_resample_audio] Resampling from {orig_sr} Hz to {target_sr} Hz using TorchaudioHannResampler..."
        )
        resampler = TorchaudioHannResampler(orig_sr, target_sr)
        audio_data = resampler.resample(audio_data)

    return audio_data


def calculate_overlap(seg1: tuple, seg2: tuple) -> float:
    start = max(seg1[0], seg2[0])
    end = min(seg1[1], seg2[1])
    return max(0.0, end - start)


def evaluate_vad_frame_based(
    labeled_segments: list,
    final_segments: list,
    audio_len_sec: float,
    resolution_ms: int = 10,
) -> None:
    """Calculates precision, recall, and overall F1 benchmarks on a frame basis (10ms bins)."""
    num_frames = int(np.ceil(audio_len_sec * 1000 / resolution_ms))

    gt_array = np.zeros(num_frames, dtype=bool)
    for start, end in labeled_segments:
        start_frame = int(start * 1000 / resolution_ms)
        end_frame = int(end * 1000 / resolution_ms)
        gt_array[start_frame:end_frame] = True

    det_array = np.zeros(num_frames, dtype=bool)
    for start, end in final_segments:
        start_frame = int(start * 1000 / resolution_ms)
        end_frame = int(end * 1000 / resolution_ms)
        det_array[start_frame:end_frame] = True

    tp = np.sum(gt_array & det_array)
    fp = np.sum(~gt_array & det_array)
    fn = np.sum(gt_array & ~det_array)
    tn = np.sum(~gt_array & ~det_array)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * (precision * recall) / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    mr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    print("--- Frame-based Evaluation (10ms resolution) ---")
    print(f"Precision:              {precision * 100:.1f}%")
    print(f"Recall (Coverage):      {recall * 100:.1f}%")
    print(f"F1-Score:               {f1 * 100:.1f}%")
    print(f"False Alarm Rate (FAR): {far * 100:.1f}%")
    print(f"Miss Rate (MR):         {mr * 100:.1f}%\n")


def evaluate_vad_performance(
    labeled_segments: list, final_segments: list
) -> None:
    """Compares target pipeline outcomes against ground truth benchmarks."""
    total_gt_duration = sum(end - start for start, end in labeled_segments)
    total_det_duration = sum(end - start for start, end in final_segments)
    total_overlap = 0.0

    matched_det = set()

    for i, gt_seg in enumerate(labeled_segments):
        print(f"Ground Truth {i} ({gt_seg[0]:.2f}s - {gt_seg[1]:.2f}s):")
        seg_overlap = 0.0
        for j, det_seg in enumerate(final_segments):
            overlap = calculate_overlap(gt_seg, det_seg)
            if overlap > 0:
                matched_det.add(j)
                diff_start = det_seg[0] - gt_seg[0]
                diff_end = det_seg[1] - gt_seg[1]

                start_str = (
                    f"{abs(diff_start):.2f}s {'late' if diff_start > 0 else 'early'}"
                    if diff_start != 0
                    else "exact"
                )
                end_str = (
                    f"{abs(diff_end):.2f}s {'late' if diff_end > 0 else 'early'}"
                    if diff_end != 0
                    else "exact"
                )

                print(
                    f"  -> Overlaps with Detected {j} ({det_seg[0]:.2f}s - {det_seg[1]:.2f}s)"
                )
                print(f"     Difference: Started {start_str}, Ended {end_str}")

                seg_overlap += overlap
                total_overlap += overlap

        gt_len = gt_seg[1] - gt_seg[0]
        print(
            f"  Coverage: {(seg_overlap / gt_len) * 100:.1f}% of this segment detected.\n"
        )

    unmatched = [j for j in range(len(final_segments)) if j not in matched_det]
    if unmatched:
        print("--- False Positives (Detected but no Ground Truth) ---")
        for j in unmatched:
            print(
                f"  Detected {j}: {final_segments[j][0]:.2f}s - {final_segments[j][1]:.2f}s"
            )
        print()

    recall = (
        (total_overlap / total_gt_duration) * 100
        if total_gt_duration > 0
        else 0.0
    )
    precision = (
        (total_overlap / total_det_duration) * 100
        if total_det_duration > 0
        else 0.0
    )

    print(
        f"Overall Recall/Coverage: {recall:.1f}% of actual speech was detected."
    )
    print(
        f"Overall Precision: {precision:.1f}% of detected speech was actual speech.\n"
    )


def visualize_comparison_with_segments(
    audio_array: np.ndarray,
    labeled_segments: list,
    final_segments: list,
    target_sr: int = TARGET_SAMPLE_RATE,
    container_id: str = "comparison_waveform",
) -> None:
    """Presents comparison waveforms matching pipeline output boundaries to Ground Truth markers."""
    wav_io = io.BytesIO()
    with wave.open(wav_io, "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(target_sr)
        wav_file.writeframes(
            (
                np.clip(
                    audio_array * INT16_MAX_FLOAT,
                    -INT16_MAX_FLOAT,
                    INT16_MAX_FLOAT - 1,
                )
            )
            .astype(np.int16)
            .tobytes()
        )
    wav_io.seek(0)
    audio_b64 = base64.b64encode(wav_io.read()).decode("utf-8")

    regions_js = ""
    for i, (s, e) in enumerate(labeled_segments):
        regions_js += f"wsRegions.addRegion({{start: {s}, end: {e}, content: 'GT {i}', color: 'rgba(50, 205, 50, 0.4)'}});\n"

    for i, (s, e) in enumerate(final_segments):
        regions_js += f"wsRegions.addRegion({{start: {s}, end: {e}, content: 'Det {i}', color: 'rgba(138, 43, 226, 0.4)'}});\n"

    html_code = f"""
    <div id='{container_id}' style='margin-top: 20px; border: 1px solid #ddd; border-radius: 4px;'></div>
    <div id='timeline-{container_id}'></div>
    <div style='margin-top: 10px; display: flex; align-items: center; gap: 15px;'>
        <button id='btn-play-{container_id}' style='padding: 8px 16px; cursor: pointer;'>Play / Pause</button>
        <span style='font-family: sans-serif; color: #555; font-size: 14px;'>Zoom: <input type="range" id="zoom-{container_id}" min="10" max="1000" value="10" style="width: 150px; vertical-align: middle;"></span>
        <span style='font-family: sans-serif; color: #555; margin-left: auto;'>Comparison: Ground Truth vs Detected</span>
    </div>
    <script type='module'>
        import WaveSurfer from 'https://unpkg.com/wavesurfer.js@7/dist/wavesurfer.esm.js';
        import RegionsPlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/regions.esm.js';
        import TimelinePlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/timeline.esm.js';
        import HoverPlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/hover.esm.js';

        const ws = WaveSurfer.create({{
            container: '#{container_id}',
            waveColor: 'tomato',
            progressColor: 'firebrick',
            height: 150,
            normalize: true,
            minPxPerSec: 10,
        }});

        const wsRegions = ws.registerPlugin(RegionsPlugin.create());
        const wsTimeline = ws.registerPlugin(TimelinePlugin.create({{
            container: '#timeline-{container_id}',
            height: 24,
            style: {{
                fontSize: '12px',
                color: '#666',
            }}
        }}));

        ws.registerPlugin(HoverPlugin.create({{
            lineColor: '#ff0000',
            lineWidth: 2,
            labelBackground: '#555',
            labelColor: '#fff',
            labelSize: '11px',
            formatTimeCallback: (sec) => sec.toFixed(3) + 's'
        }}));

        ws.load('data:audio/wav;base64,{audio_b64}');
        ws.on('decode', () => {{
            {regions_js}
            const slider = document.getElementById('zoom-{container_id}');
            slider.addEventListener('input', (e) => {{
                ws.zoom(e.target.valueAsNumber);
            }});
        }});
        document.getElementById('btn-play-{container_id}').onclick = () => ws.playPause();
    </script>
    """
    display(HTML(html_code))


def visualize_audio_with_segments(
    audio_array: np.ndarray,
    segments: list,
    target_sr: int = TARGET_SAMPLE_RATE,
    container_id: str = "waveform",
    wave_color: str = "tomato",
    progress_color: str = "firebrick",
    region_color: str = "rgba(255, 99, 71, 0.4)",
    title_text: str = "Audio Configuration",
) -> None:
    """Creates interactable waveform charts representing slice regions natively via WaveSurfer.js."""
    wav_io = io.BytesIO()
    with wave.open(wav_io, "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(target_sr)
        wav_file.writeframes(
            (
                np.clip(
                    audio_array * INT16_MAX_FLOAT,
                    -INT16_MAX_FLOAT,
                    INT16_MAX_FLOAT - 1,
                )
            )
            .astype(np.int16)
            .tobytes()
        )
    wav_io.seek(0)
    audio_b64 = base64.b64encode(wav_io.read()).decode("utf-8")

    regions_js = "".join(
        [
            f"wsRegions.addRegion({{start: {s}, end: {e}, content: 'Seg {i}', color: '{region_color}'}});\n"
            for i, (s, e) in enumerate(segments)
        ]
    )

    html_code = f"""
    <div id='{container_id}' style='margin-top: 20px; border: 1px solid #ddd; border-radius: 4px;'></div>
    <div id='timeline-{container_id}'></div>
    <div style='margin-top: 10px; display: flex; align-items: center; gap: 15px;'>
        <button id='btn-play-{container_id}' style='padding: 8px 16px; cursor: pointer;'>Play / Pause</button>
        <span style='font-family: sans-serif; color: #555; font-size: 14px;'>Zoom: <input type="range" id="zoom-{container_id}" min="10" max="1000" value="10" style="width: 150px; vertical-align: middle;"></span>
        <span style='font-family: sans-serif; color: #555; margin-left: auto;'>{title_text} ({len(segments)} segments)</span>
    </div>
    <script type='module'>
        import WaveSurfer from 'https://unpkg.com/wavesurfer.js@7/dist/wavesurfer.esm.js';
        import RegionsPlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/regions.esm.js';
        import TimelinePlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/timeline.esm.js';
        import HoverPlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/hover.esm.js';

        const ws = WaveSurfer.create({{
            container: '#{container_id}',
            waveColor: '{wave_color}',
            progressColor: '{progress_color}',
            height: 150,
            normalize: true,
            minPxPerSec: 10,
        }});

        const wsRegions = ws.registerPlugin(RegionsPlugin.create());
        const wsTimeline = ws.registerPlugin(TimelinePlugin.create({{
            container: '#timeline-{container_id}',
            height: 24,
            style: {{
                fontSize: '12px',
                color: '#666',
            }}
        }}));

        ws.registerPlugin(HoverPlugin.create({{
            lineColor: '#ff0000',
            lineWidth: 2,
            labelBackground: '#555',
            labelColor: '#fff',
            labelSize: '11px',
            formatTimeCallback: (sec) => sec.toFixed(3) + 's'
        }}));

        ws.load('data:audio/wav;base64,{audio_b64}');
        ws.on('decode', () => {{
            {regions_js}
            const slider = document.getElementById('zoom-{container_id}');
            slider.addEventListener('input', (e) => {{
                ws.zoom(e.target.valueAsNumber);
            }});
        }});
        document.getElementById('btn-play-{container_id}').onclick = () => ws.playPause();
    </script>
    """
    display(HTML(html_code))

In [ ]:
# @title Select or Upload Audio File

# fmt: off
# Option A: Specify a local audio filepath directly
local_audio_path = "backend/pipeline/segmentation/tests/test_data/test_dispatch_amador.flac"  # @param {type:"string"}
# fmt: on

if local_audio_path and os.path.exists(local_audio_path):
    AUDIO_FILEPATH = str(Path(local_audio_path).resolve())
    print(f"Using local file: {AUDIO_FILEPATH}")
elif HAS_COLAB_FILES:
    print("Please select an audio file to upload:\n")
    temp_dir = tempfile.mkdtemp()
    uploaded = files.upload(target_dir=temp_dir)
    if uploaded:
        filename = list(uploaded.keys())[0]
        AUDIO_FILEPATH = os.path.join(temp_dir, filename)
        print(f"\nSuccess! AUDIO_FILEPATH set to: {AUDIO_FILEPATH}")
    else:
        AUDIO_FILEPATH = None
        print("\nNo file uploaded.")
else:
    AUDIO_FILEPATH = None
    print(
        f"File '{local_audio_path}' not found and Colab file uploader not available."
    )

In [ ]:
# @title Bandpass + UL-UNAS (Dry/Wet) + EQ + Production VAD
# fmt: off
# @markdown ### Bandpass Settings
highpass_hz = 300  # @param {type:"slider", min:50, max:1000, step:50}
lowpass_hz = 4000  # @param {type:"slider", min:2000, max:8000, step:100}

# @markdown ### Denoiser Settings
blend_ratio = 0.8  # @param {type:"slider", min:0.0, max:1.0, step:0.05}

# @markdown ### EQ / Presence Boost Settings
boost_freq_hz = 2500  # @param {type:"slider", min:1000, max:4000, step:100}
boost_gain_db = 10  # @param {type:"slider", min:0.0, max:24.0, step:1.0}
peak_filter_q = 1.0  # @param {type:"slider", min:0.1, max:5.0, step:0.1}

# @markdown ### VAD (Voice Activity Detection) Settings
vad_threshold_onset = 0.20  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
vad_threshold_offset = 0.20  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
min_speech_duration_ms = 150  # @param {type:"slider", min:50, max:1000, step:50}
min_silence_duration_ms = 750  # @param {type:"slider", min:100, max:2000, step:100}
pad_sec = 0.3  # @param {type:"slider", min:0.0, max:1.0, step:0.1}
# fmt: on

if "AUDIO_FILEPATH" in locals() and AUDIO_FILEPATH:
    print("1. Loading and Resampling Audio...")
    audio = load_and_resample_audio(
        AUDIO_FILEPATH, target_sr=TARGET_SAMPLE_RATE
    )

    print(
        "2. Instantiating Production VoiceActivityDetector with Colab Settings..."
    )
    detector = VoiceActivityDetector(
        highpass_hz=float(highpass_hz),
        lowpass_hz=float(lowpass_hz),
        blend_ratio=float(blend_ratio),
        boost_freq_hz=float(boost_freq_hz),
        boost_gain_db=float(boost_gain_db),
        peak_filter_q=float(peak_filter_q),
        threshold_onset=float(vad_threshold_onset),
        threshold_offset=float(vad_threshold_offset),
        min_speech_duration_ms=int(min_speech_duration_ms),
        min_silence_duration_ms=int(min_silence_duration_ms),
        pad_sec=float(pad_sec),
    )

    print("3. Executing Production Speech Detection...")
    final_segments, preprocessed_audio = detector.detect_speech_segments(audio)

    # Use preprocessed audio (or raw audio if None) for playback & visualization
    final_audio = (
        preprocessed_audio if preprocessed_audio is not None else audio
    )

    print(f"\nFinished! Found {len(final_segments)} speech segments.")

    print("\nPlayback:")
    display(Audio(final_audio, rate=TARGET_SAMPLE_RATE, normalize=False))

    print("\nVisualizing results...")
    visualize_audio_with_segments(
        audio_array=final_audio,
        segments=final_segments,
        target_sr=TARGET_SAMPLE_RATE,
        title_text=f"Production VAD Pipeline (Onset: {vad_threshold_onset}, Offset: {vad_threshold_offset}, Pad: {pad_sec}s)",
        region_color="rgba(138, 43, 226, 0.4)",
    )
else:
    print("Please select or upload an audio file first in Cell 6.")

In [ ]:
# @title VAD confidence plot

if "final_audio" in locals() and "detector" in locals():
    print(
        "Calculating VAD probabilities over time via production Silero session..."
    )
    sr_tensor = np.array([TARGET_SAMPLE_RATE], dtype=np.int64)

    state = np.zeros((2, 1, 128), dtype=np.float32)
    context = np.zeros(64, dtype=np.float32)

    probabilities = []
    times = []

    for i in range(0, len(final_audio), 512):
        chunk = final_audio[i : i + 512]
        if len(chunk) < 512:
            chunk = np.pad(chunk, (0, 512 - len(chunk)))

        x_with_context = np.concatenate([context, chunk])
        ort_inputs = {
            "input": x_with_context.reshape(1, 576).astype(np.float32),
            "state": state,
            "sr": sr_tensor,
        }

        outputs = detector.silero_session.run(None, ort_inputs)
        prob = float(outputs[0].flatten()[0])
        state = outputs[1]
        context = x_with_context[-64:]

        probabilities.append(prob)
        times.append(i / TARGET_SAMPLE_RATE)

    plt.figure(figsize=(15, 4))
    plt.plot(times, probabilities, label="VAD Probability", color="purple")
    plt.axhline(
        y=vad_threshold_offset,
        color="r",
        linestyle="--",
        label=f"Offset Threshold ({vad_threshold_offset})",
    )
    plt.axhline(
        y=vad_threshold_onset,
        color="g",
        linestyle="--",
        label=f"Onset Threshold ({vad_threshold_onset})",
    )

    time_axis = np.linspace(
        0, len(final_audio) / TARGET_SAMPLE_RATE, num=len(final_audio)
    )
    max_peak = np.max(np.abs(final_audio)) if len(final_audio) > 0 else 1.0
    if max_peak > 0:
        plt.plot(
            time_axis,
            np.abs(final_audio) / max_peak * 0.5,
            color="gray",
            alpha=0.3,
            label="Audio Envelope",
        )

    plt.xlim(0, len(final_audio) / TARGET_SAMPLE_RATE)
    plt.title("Silero VAD Confidence vs Time (Production Pipeline)")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Probability")
    plt.legend(loc="upper right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Please execute Cell 7 first.")

In [ ]:
# @title VAD Pipeline vs. Ground Truth Comparison

if "AUDIO_FILEPATH" in locals() and AUDIO_FILEPATH:
    file_name = pathlib.Path(AUDIO_FILEPATH).name
    if file_name not in GROUND_TRUTH_SEGMENTS:
        print(
            f"No ground truth labels found for '{file_name}'. Skipping evaluation."
        )
    else:
        labeled_segments = GROUND_TRUTH_SEGMENTS[file_name]

        if "final_segments" in locals() and "labeled_segments" in locals():
            evaluate_vad_performance(labeled_segments, final_segments)

            if "final_audio" in locals():
                audio_len_sec = len(final_audio) / TARGET_SAMPLE_RATE
                evaluate_vad_frame_based(
                    labeled_segments, final_segments, audio_len_sec
                )

                print(
                    "Visualizing Comparison (Green = Ground Truth, Purple = Detected)..."
                )
                visualize_comparison_with_segments(
                    final_audio, labeled_segments, final_segments
                )
        else:
            print(
                "Make sure both 'final_segments' and 'labeled_segments' are defined."
            )
else:
    print("Please select or upload an audio file first.")